```sql
SELECT
    u.username,
    o.id AS order_id,
    o.amount
FROM users AS u
JOIN orders AS o
    ON o.user_id = u.id;
```
`ON o.user_id = u.id`：订单的 user_id 等于用户的 id 时，两行可以组合。

用别名是因为2张表可能都都有一样的字段，比如id。

### INNER JOIN

```sql
SELECT
    u.username,
    o.id,
    o.amount
FROM users AS u
INNER JOIN orders AS o
    ON o.user_id = u.id;
```
inner可以省略，可以理解为`只要交集`。

**重要规则：JOIN 的结果是一组匹配组合，不保证左表的一行在结果中只出现一次。**

### LEFT JOIN
LEFT JOIN 保证：左表中的每一行至少出现一次。右表匹配不到时，右侧字段补成 NULL。

### RIGHT JOIN
保证右表中的所有行出现。逻辑上可以通过交换表顺序改写为 LEFT JOIN

### FULL JOIN
```sql
SELECT ...
FROM users AS u
FULL JOIN orders AS o
    ON o.user_id = u.id;
```
它会保留：
- 匹配的用户与订单
- 没有订单的用户
- 没有用户的订单

### CROSS JOIN
笛卡尔积，即a表条数*b表条数的组合

适合：
- 生成所有颜色和尺码组合
- 生成日期和类别组合
- 构造测试数据

### ON 和 WHERE 的区别

#### on
保留所有用户，但右侧只匹配 paid 订单。
```sql
SELECT
    u.username,
    o.id,
    o.status
FROM users AS u
LEFT JOIN orders AS o
    ON o.user_id = u.id
   AND o.status = 'paid';
```

### WHERE
过滤右表
```sql
SELECT
    u.username,
    o.id,
    o.status
FROM users AS u
LEFT JOIN orders AS o
    ON o.user_id = u.id
WHERE o.status = 'paid';
```

- ON：决定右表中的哪些行能够参与匹配
- WHERE：连接完成后，再过滤整个结果集

### 多表 JOIN
假设还有其他表：
```
users
orders
order_items
products
```

```sql
SELECT
    u.username,
    o.id AS order_id,
    p.name AS product_name,
    oi.quantity,
    oi.unit_price
FROM users AS u
JOIN orders AS o
    ON o.user_id = u.id
JOIN order_items AS oi
    ON oi.order_id = o.id
JOIN products AS p
    ON p.id = oi.product_id
WHERE o.id = 101;
```
```
users.id
    ↓
orders.user_id

orders.id
    ↓
order_items.order_id

products.id
    ↓
order_items.product_id
```

### 多次一对多导致行数继续扩张
用户 Alice 有 2 张订单,每张订单有 3 个订单项
```sql
users
JOIN orders
JOIN order_items
```
所以查询结果“重复”时，应先判断：是真实重复数据，还是多个一对多关系展开后的组合？


- INNER JOIN 只保留匹配行。
- LEFT JOIN 保留左表所有行。
- 一对多 JOIN 会让左表记录出现多次。
- ON 决定如何匹配，WHERE 过滤连接后的结果。
- LEFT JOIN 的右表过滤条件放错到 WHERE，可能使它接近 INNER JOIN。
- LEFT JOIN 统计右表数量时通常使用 COUNT(right_table.id)。
- 外键保证引用完整性，JOIN 负责查询组合。
多次一对多 JOIN 可能造成结果行数快速膨胀。